# 2. The regime model

How many regimes the data support, what they look like, and how long the model
keeps knowing anything about them. Stage gates two and three.

Run `forecast fit-regimes` and `forecast forecast-now` before this notebook; both
write what it reads. All computation lives in the package.

In [ ]:
from datetime import date

import pandas as pd

from economic_regime_forecasting import pipeline_gates
from economic_regime_forecasting.configuration.registry import load_registries
from economic_regime_forecasting.configuration.run_settings import (
    ARTIFACTS,
    DEFAULT_RUN_SETTINGS,
)
from economic_regime_forecasting.data.cache import ArtifactStore, SeriesCache
from economic_regime_forecasting.data.panel import assemble_point_in_time_panel, load_final_series
from economic_regime_forecasting.features.observation_matrix import build_observation_matrix
from economic_regime_forecasting.models.gaussian_hidden_markov_model import (
    GaussianHiddenMarkovModel,
)
from economic_regime_forecasting.models.regime_forecast import (
    measure_mixing,
    transition_matrix_table,
)
from economic_regime_forecasting.models.state_labelling import describe_regimes, regime_table
from economic_regime_forecasting.reporting import figures

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

settings = DEFAULT_RUN_SETTINGS
registry, indicators = load_registries()
cache = SeriesCache(settings.cache.raw, settings.cache.vintage)
artifacts = ArtifactStore(settings.cache.models)
today = date.today()

matrix = build_observation_matrix(assemble_point_in_time_panel(registry, today, cache), registry)
model = GaussianHiddenMarkovModel.from_dictionary(artifacts.read_json(ARTIFACTS.selected_model))
print(matrix.describe(), "|", model.state_count, "regimes")

## How many regimes?

Persistence and population are hard floors: a state lasting under three months on
average is noise wearing a label, and one holding under five percent of months
cannot support a conditional base rate. Among the candidates that clear both, the
held-out log likelihood decides, because it asks the out-of-sample question
directly. The information criterion is reported alongside, and a disagreement
between them is reported rather than resolved quietly.

In [ ]:
sweep = artifacts.read_table(ARTIFACTS.state_count_sweep)
sweep.round(4)

## What the regimes are

Emission means in natural units, so a reader can recognise the economy being
described rather than take a z-score on trust. State zero is always the weakest
growth environment the model found; that ordering is imposed after fitting so it
means the same thing in every refit of the backtest.

In [ ]:
regimes = describe_regimes(model, matrix.values, matrix.transformed.to_numpy())
descriptions = regime_table(regimes)
descriptions.round(3)

In [ ]:
figures.plot_regime_means(descriptions)

## The transition matrix

Rows are where the economy is, columns are where it goes next month. The strong
diagonal is the whole story: these are regimes, not weather.

In [ ]:
labels = [f"{item.state}: {item.label}" for item in regimes]
transition_matrix_table(model, labels).round(4)

## Where the model thinks we have been

Filtered probabilities, which condition on the past and present and never on the
future. National Bureau of Economic Research recessions are shaded for reference;
they are not an input to the model and never have been.

In [ ]:
filtered = model.filtered_state_probabilities(matrix.values)
recession = load_final_series(registry, cache, ["recession_indicator"])["recession_indicator"]
figures.plot_regime_probabilities(
    matrix.dates, filtered, [item.label for item in regimes], recession
)

## Gate 2

In [ ]:
from economic_regime_forecasting.models.state_selection import (
    StateCountEvaluation,
    StateCountSweep,
)

evaluations = tuple(
    StateCountEvaluation(
        state_count=int(row["states"]),
        free_parameters=int(row["free_parameters"]),
        training_log_likelihood=float(row["training_log_likelihood"]),
        bayesian_information_criterion=float(row["bayesian_information_criterion"]),
        held_out_log_likelihood_per_month=float(row["held_out_log_likelihood_per_month"]),
        smallest_population_share=float(row["smallest_population_share"]),
        shortest_expected_duration_in_months=float(row["shortest_expected_duration_months"]),
        second_largest_eigenvalue_modulus=float(row["second_eigenvalue_modulus"]),
        converged=True,
    )
    for row in sweep.to_dict("records")
)
rebuilt = StateCountSweep(
    evaluations=evaluations,
    models={model.state_count: model},
    recommended_state_count=model.state_count,
    reason="loaded from the fitted artifact",
    runner_up_state_count=None,
)
print(
    pipeline_gates.gate_two_regime_model(
        rebuilt, model.most_likely_state_path(matrix.values), len(matrix)
    ).describe()
)

## The information horizon

A transition matrix mixes. The distance between a projected regime distribution
and the model's long-run distribution decays like the second largest eigenvalue
modulus raised to the horizon. Past some point the projection *is* the
unconditional base rate, and presenting it as a prediction would misdescribe it
even if it scored well.

Where the curve below crosses the threshold is that point.

In [ ]:
mixing = measure_mixing(
    model,
    filtered,
    (12, 24, 60, 120, 240),
    settings.information_horizon_total_variation_threshold,
)
print(mixing.describe())
mixing.table().round(4)

In [ ]:
figures.plot_mixing(mixing.table(), settings.information_horizon_total_variation_threshold)

## The current forecast grid

Ten indicators by three horizons, with the evidence behind each. `effective sample
size` is how many months of history stand behind that particular number;
`distance to stationary` is how far the projection still is from the base rate.

In [ ]:
forecasts = artifacts.read_table(ARTIFACTS.current_forecasts)
forecasts.pivot(index="indicator", columns="horizon_months", values="probability").round(3)

In [ ]:
forecasts[
    [
        "indicator",
        "horizon_months",
        "probability",
        "composition",
        "effective_sample_size",
        "distance_to_stationary",
    ]
].round(3)

## Gate 3

In [ ]:
print(
    pipeline_gates.gate_three_forecasts(
        forecasts, list(indicators), settings.forecast_horizons_in_months, mixing
    ).describe()
)